# vislens — two-tower fine-tune

A launcher, not logic (`CLAUDE.md` section 10). Everything it calls lives in the repo at a
**pinned SHA**, so this notebook plus that SHA reproduces the run exactly. If you find
yourself editing a cell below to change behaviour, change the repo and re-pin instead.

**Before running:** Settings -> Accelerator -> **GPU T4 x2**, and Internet **on**.
The P100 will not work — Kaggle's own torch build excludes `sm_60`, so you get
`no kernel image is available for execution`.


In [ ]:
SHA = 'edd0a32c7b5b54bcb1f04c5c111ccd3688e88844'  # pin: change this, not the cells below
!pip install -q git+https://github.com/Het415/vislens@{SHA}


In [ ]:
import torch

assert torch.cuda.is_available(), 'no GPU — set Accelerator to GPU T4 x2'
name = torch.cuda.get_device_name(0)
print(name, '|', torch.__version__)
# sm_75 is the T4. A P100 is sm_60 and Kaggle's torch has no kernels for it, so fail
# here with a readable message rather than 300 steps in with a CUDA error.
assert 'T4' in name, f'expected a T4, got {name!r} — switch the accelerator'


In [ ]:
from vislens.data.text_emb import load_split_embeddings
from vislens.models.towers import TowerConfig, build_two_tower
from vislens.train.config import TrainConfig
from vislens.train.loop import train

DATA = '/kaggle/input/vislens-shards'

# Both splits in one lookup: the val loader shares the training collate, and a
# train-only table drops every val batch -> NaN val_loss -> no best.pt.
lookup, text = load_split_embeddings(f'{DATA}/text_emb', ('train', 'val'))
print(f'{len(lookup):,} text embeddings, dim {text.shape[1]}')


In [ ]:
config = TrainConfig(
    shards_dir=f'{DATA}/shards',
    runs_dir='/kaggle/working/runs',
    batch_size=256,
    epochs=4,
    num_workers=2,       # Kaggle gives 4 vCPU; 2 workers leaves room for the main process
    max_minutes=680,     # exit at ~11h20m, inside Kaggle's 12h kill, on our own terms
)
print('run_id:', config.run_id)

model = build_two_tower(TowerConfig(), text_dim=text.shape[1], device='cuda')
print(f'{sum(p.numel() for p in model.parameters())/1e6:.1f}M params')


In [ ]:
summary = train(model, config, lookup, text, device='cuda')
summary


In [ ]:
# Two checkpoints and the metrics, nothing per-step: Kaggle caps a notebook at ~500
# output files, and the run record is what gets committed back to the repo.
!ls -la /kaggle/working/runs/*/
!head -5 /kaggle/working/runs/*/metrics.csv
